# Elhub 2021 → Cassandra → Spark → Plots → MongoDB

This notebook performs the entire task:

1. **Fetch** hourly production data for *all* price areas from Elhub `PRODUCTION_PER_GROUP_MBA_HOUR` for the **entire year 2021**, month by month (Usage Guidelines: one month per call, **inclusive** start and end, fixed offset `+00:00` to avoid DST errors).
2. **Extract** only the `productionPerGroupMbaHour` list, normalize to `priceArea`, `productionGroup`, `startTime`, `quantityKwh` and create a **Spark DataFrame**.
3. **Write** to the Cassandra table `elhub_data.production_hourly_by_group` (PK ((pricearea, productiongroup), starttime)).
4. **Read** the same 4 columns from Cassandra.
5. **Plot**:
   - Pie: total production for the year for a selected price area (one slice per group).
   - Line: first month for the selected price area, one line per group.
6. **Save** Spark data to **MongoDB Atlas**.

In [3]:
# === Parameters and Configuration ===
# --- General Time Constraints ---
YEAR = 2021  # Target year for data retrieval and analysis (Elhub API calls)
CHOSEN_AREA = "NO1"  # Price area selected for demonstration plots in the notebook
FIRST_MONTH = 1  # Month selected for time-series line plot (January)

# --- Cassandra Database Configuration ---
CASSANDRA_HOST = "127.0.0.1"  # Local host address for the Cassandra cluster
CASSANDRA_PORT = "9042"  # Default port for Cassandra
CASSANDRA_KEYSPACE = "elhub_data"  # Keyspace used to store the Elhub data
CASSANDRA_TABLE = "production_hourly_by_group"  # Table containing the raw hourly production data

# --- MongoDB (Atlas) Database Configuration ---
# NOTE: The sensitive password/user info should ideally be loaded from a secrets file or environment variable.
MONGO_URI = (
    "mongodb+srv://AHS_db_user:"
    "fAdp5LC8GglDRedl"  # Replace with actual password/secret in a secure setup
    "@ahs786student.qh8rsrb.mongodb.net/elhub"
    "?retryWrites=true&w=majority&tls=true&appName=AHS786Student"
)
MONGO_DB = "elhub"  # Database name where the curated data is stored
MONGO_COLL = "production_2021_hourly_by_group"  # Collection name for the time-series production data

# --- Elhub API Details ---
BASE_URL = "https://api.elhub.no/energy-data/v0/price-areas"  # Base URL for the Elhub Price Area API endpoint
DATASET  = "PRODUCTION_PER_GROUP_MBA_HOUR"  # Specific dataset requested: hourly production by group for each MBA

In [4]:
# === SparkSession with Cassandra and Mongo Connectors ===

from pyspark.sql import SparkSession

# Define the required packages for the connectors
cassandra_pkg = "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1"
mongo_pkg     = "org.mongodb.spark:mongo-spark-connector_2.12:10.5.0"

# Build and configure the SparkSession
spark = (
    SparkSession.builder
    # Set a descriptive application name for tracking in the Spark UI
    .appName("ELHUB_2021_to_Cassandra_Mongo_Plots")
    # Use all available cores locally ("local[*]")
    .master("local[*]")
    # Ensure all time-related operations and data are treated as UTC to match Elhub API
    .config("spark.sql.session.timeZone", "UTC")
    # Specify the external JAR packages needed for Cassandra and MongoDB integration
    .config("spark.jars.packages", f"{cassandra_pkg},{mongo_pkg}")
    # Configure the host address for the Cassandra cluster connection
    .config("spark.cassandra.connection.host", CASSANDRA_HOST)
    # Configure the port for the Cassandra connection
    .config("spark.cassandra.connection.port", CASSANDRA_PORT)
    # Force the Spark driver to use TLSv1.2 (often required for secure connections)
    .config("spark.driver.extraJavaOptions", "-Djdk.tls.client.protocols=TLSv1.2")
    # Force the Spark executors to use TLSv1.2
    .config("spark.executor.extraJavaOptions", "-Djdk.tls.client.protocols=TLSv1.2")
    # Create or return the existing SparkSession
    .getOrCreate()
)

# Display the SparkSession object for confirmation (standard practice in notebooks)
spark

In [5]:
# === Helper Functions ===
import requests, time, calendar
from typing import Dict, List, Tuple
from datetime import datetime, timezone
from pyspark.sql import functions as F, types as T

def month_utc_range(year: int, month: int) -> Tuple[str, str]:
    """
    Calculates the start and end timestamps (inclusive) for a given month in UTC,
    formatted specifically for the Elhub API query.

    The API requires a fixed +00:00 offset, even if the input time is technically
    local, to avoid Daylight Saving Time (DST) related errors.
    """
    # Start of the month (local assumption for calculation simplicity)
    start_local = datetime(year, month, 1, 0, 0, 0)
    # Determine the last day of the month
    last_day = calendar.monthrange(year, month)[1]
    # End of the month (23:00 on the last day, as per the original implementation)
    # NOTE: Elhub API often expects the full hour, up to 23:00.
    end_local = datetime(year, month, last_day, 23, 0, 0)
    
    # Explicitly set timezone to UTC
    start_utc = start_local.replace(tzinfo=timezone.utc)
    end_utc   = end_local.replace(tzinfo=timezone.utc)
    
    # Format string: ISO 8601 with fixed UTC offset
    fmt = "%Y-%m-%dT%H:%M:%S+00:00"
    return start_utc.strftime(fmt), end_utc.strftime(fmt)

def _merge_with_area(items: List[Dict], pa_code: str) -> List[Dict]:
    """
    Internal helper to inject the 'priceArea' code into each item in a list
    if it is missing, based on the parent structure's code.
    """
    out: List[Dict] = []
    for it in items:
        if isinstance(it, dict):
            # If 'priceArea' key is missing, add it
            if "priceArea" not in it and pa_code:
                # Create a copy before modifying
                it = dict(it)
                it["priceArea"] = pa_code
            out.append(it)
    return out

def _extract_prod_list(js: Dict) -> List[Dict]:
    """
    Recursively extracts the 'productionPerGroupMbaHour' list from the complex
    JSON structure returned by the Elhub API, handling various nesting levels.
    It also attempts to handle price area codes when they are nested.
    """
    # 1. Direct key lookup (simple case)
    items = js.get("productionPerGroupMbaHour")
    if isinstance(items, list): return items
    
    # 2. Key nested under 'data'
    data = js.get("data")
    if isinstance(data, dict):
        items = data.get("productionPerGroupMbaHour")
        if isinstance(items, list): return items
        
        # 3. Data nested under 'priceAreas' list inside 'data'
        pa_list = data.get("priceAreas")
        if isinstance(pa_list, list):
            merged: List[Dict] = []
            for pa in pa_list:
                if not isinstance(pa, dict): continue
                # Find the price area code from common keys
                pa_code = pa.get("priceArea") or pa.get("code") or pa.get("name")
                # Find the list of production items
                li = pa.get("productionPerGroupMbaHour") or (pa.get("attributes", {}) if isinstance(pa.get("attributes", {}), dict) else {}).get("productionPerGroupMbaHour")
                if isinstance(li, list): merged.extend(_merge_with_area(li, pa_code))
            if merged: return merged

    # 4. Data nested under 'attributes' in a list under 'data'
    if isinstance(data, list):
        merged: List[Dict] = []
        for obj in data:
            if not isinstance(obj, dict): continue
            attrs = obj.get("attributes", {}) if isinstance(obj.get("attributes", {}), dict) else {}
            pa_code = obj.get("priceArea") or attrs.get("priceArea")
            li = obj.get("productionPerGroupMbaHour") or attrs.get("productionPerGroupMbaHour")
            if isinstance(li, list):
                # Simple case: production list found at this level
                merged.extend(_merge_with_area(li, pa_code)); continue
            
            # Complex case: nested 'priceAreas' list under 'attributes'
            pa_list = attrs.get("priceAreas")
            if isinstance(pa_list, list):
                for pa in pa_list:
                    if not isinstance(pa, dict): continue
                    pa_code2 = pa.get("priceArea") or pa.get("code") or pa.get("name")
                    li2 = pa.get("productionPerGroupMbaHour") or (pa.get("attributes", {}) if isinstance(pa.get("attributes", {}), dict) else {}).get("productionPerGroupMbaHour")
                    if isinstance(li2, list): merged.extend(_merge_with_area(li2, pa_code2))
        if merged: return merged

    # If extraction failed, raise an error with debug info
    top_keys = list(js.keys())
    dtype = type(data).__name__
    sample = None
    if isinstance(data, list) and data: sample = list(data[0].keys())
    raise RuntimeError(f"Could not find 'productionPerGroupMbaHour'. Top keys={top_keys}, data type={dtype}, sample[0] keys={sample}")

def fetch_all_pages(url: str, params: Dict) -> List[Dict]:
    """
    Fetches data from the Elhub API, automatically handling pagination
    by following the 'next' link until all pages are retrieved.
    """
    rows: List[Dict] = []
    # Use a session for persistent connection headers
    s = requests.Session()
    s.headers.update({"Accept":"application/json"})
    next_url, next_params = url, params.copy()
    
    while True:
        # Perform the GET request
        r = s.get(next_url, params=next_params, timeout=60)
        r.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
        js = r.json()
        
        # Extract the relevant data list
        items = _extract_prod_list(js)
        if not isinstance(items, list): 
            raise RuntimeError("'productionPerGroupMbaHour' was not a list after extraction.")
        rows.extend(items)
        
        # Check for pagination link
        links = js.get("links") or {}; nxt = links.get("next")
        
        if not nxt: 
            break # No more pages, exit loop
        
        # Prepare for the next request
        next_url, next_params = nxt, {}
        time.sleep(0.05) # Small delay to be polite to the API server
        
    return rows

def normalize_rows(items: List[Dict]) -> List[Dict]:
    """
    Standardizes the list of dictionaries to a fixed schema (4 columns), 
    handling various casing/naming conventions found in the raw API data.
    """
    out: List[Dict] = []
    for it in items:
        if not isinstance(it, dict): continue
        
        # Standardize key names by checking for common variations
        price_area = it.get("priceArea") or it.get("price_area")
        prod_group = it.get("productionGroup") or it.get("production_group")
        start_time = it.get("startTime") or it.get("start_time") or it.get("start")
        qty        = it.get("quantityKwh") or it.get("quantitykwh") or it.get("quantity")
        
        # Only include rows where all required fields are present
        if None in (price_area, prod_group, start_time, qty): continue
        
        # Append the standardized dictionary
        out.append({
            "priceArea": price_area,
            "productionGroup": prod_group,
            "startTime": start_time,
            "quantityKwh": qty
        })
    return out

In [6]:
# === Fetch 2021 Data, Write to Cassandra ===
from pyspark.sql import functions as F, types as T
total_written = 0
# Loop through all 12 months of the target year (1 to 12)
for m in range(1, 13):
    # Calculate the fixed UTC start and end timestamps for the current month
    start_iso, end_iso = month_utc_range(YEAR, m)
    
    # Define API query parameters
    params = {"dataset": DATASET, "startDate": start_iso, "endDate": end_iso}
    
    # Print status before fetching
    print(f"Fetching {YEAR}-{m:02d}  [{start_iso} → {end_iso}] ...", flush=True)
    
    # 1. EXTRACT: Fetch all pages of raw JSON data from the Elhub API
    items = fetch_all_pages(BASE_URL, params)
    
    # 2. TRANSFORM: Normalize the list of dictionaries to a standardized schema
    rows  = normalize_rows(items)
    
    # 3. TRANSFORM: Create a Spark DataFrame and cast/rename columns
    sdf = (
        spark.createDataFrame(rows)
             # Select the initial 4 columns
             .select("priceArea","productionGroup","startTime","quantityKwh")
             # Convert the string 'startTime' to a proper Spark TimestampType (required for Cassandra time column)
             .withColumn("starttime", F.to_timestamp("startTime"))
             # Rename to lowercase to match the Cassandra table schema
             .withColumn("pricearea", F.col("priceArea"))
             .withColumn("productiongroup", F.col("productionGroup"))
             # Cast the quantity to DoubleType and rename
             .withColumn("quantitykwh", F.col("quantityKwh").cast(T.DoubleType()))
             # Select the final, normalized columns for writing
             .select("pricearea","productiongroup","starttime","quantitykwh")
    )
    
    # 4. LOAD: Write the monthly DataFrame to Cassandra
    (sdf.write.format("org.apache.spark.sql.cassandra")
        .mode("append") # Use "append" since we are adding data month-by-month
        .options(keyspace=CASSANDRA_KEYSPACE, table=CASSANDRA_TABLE)
        .save())
        
    # Update total count
    n = sdf.count()
    total_written += n
    print(f"  Rows this month (after cleanup): {n:,}")

# Final confirmation of the ETL process
print(f"Done. Total rows written: {total_written:,} rows → {CASSANDRA_KEYSPACE}.{CASSANDRA_TABLE}")

Fetching 2021-01  [2021-01-01T00:00:00+00:00 → 2021-01-31T23:00:00+00:00] ...


  Rows this month (after cleanup): 13,876
Fetching 2021-02  [2021-02-01T00:00:00+00:00 → 2021-02-28T23:00:00+00:00] ...
  Rows this month (after cleanup): 13,106
Fetching 2021-03  [2021-03-01T00:00:00+00:00 → 2021-03-31T23:00:00+00:00] ...
  Rows this month (after cleanup): 15,072
Fetching 2021-04  [2021-04-01T00:00:00+00:00 → 2021-04-30T23:00:00+00:00] ...
  Rows this month (after cleanup): 15,548
Fetching 2021-05  [2021-05-01T00:00:00+00:00 → 2021-05-31T23:00:00+00:00] ...
  Rows this month (after cleanup): 16,103
Fetching 2021-06  [2021-06-01T00:00:00+00:00 → 2021-06-30T23:00:00+00:00] ...
  Rows this month (after cleanup): 15,701
Fetching 2021-07  [2021-07-01T00:00:00+00:00 → 2021-07-31T23:00:00+00:00] ...
  Rows this month (after cleanup): 16,176
Fetching 2021-08  [2021-08-01T00:00:00+00:00 → 2021-08-31T23:00:00+00:00] ...
  Rows this month (after cleanup): 16,073
Fetching 2021-09  [2021-09-01T00:00:00+00:00 → 2021-09-30T23:00:00+00:00] ...
  Rows this month (after cleanup): 15,34

In [7]:
# === Read 4 Columns from Cassandra ===

from pyspark.sql import functions as F

# Read the data from the Cassandra table using the Spark-Cassandra connector
cdf = (
    spark.read.format("org.apache.spark.sql.cassandra")
    # Specify the keyspace and table to read from
    .options(keyspace=CASSANDRA_KEYSPACE, table=CASSANDRA_TABLE)
    .load()
    # Select the four required columns (which were stored in lowercase in Cassandra)
    .select("pricearea", "productiongroup", "starttime", "quantitykwh")
    
    # Rename columns back to the camelCase style used in the Elhub API and for Pandas/Plotly consistency
    .withColumnRenamed("pricearea", "priceArea")
    .withColumnRenamed("productiongroup", "productionGroup")
    .withColumnRenamed("starttime", "startTime")
    .withColumnRenamed("quantitykwh", "quantityKwh")
)

# Display the schema to confirm data types (especially TimestampType for startTime)
cdf.printSchema()

# Show the first 5 rows of the resulting DataFrame for visual inspection
cdf.show(5, truncate=False)

root
 |-- priceArea: string (nullable = false)
 |-- productionGroup: string (nullable = false)
 |-- startTime: timestamp (nullable = true)
 |-- quantityKwh: double (nullable = true)

+---------+---------------+-------------------+-----------+
|priceArea|productionGroup|startTime          |quantityKwh|
+---------+---------------+-------------------+-----------+
|NO4      |hydro          |2021-12-31 21:00:00|3673640.2  |
|NO4      |hydro          |2021-12-31 20:00:00|3621014.0  |
|NO4      |hydro          |2021-12-31 19:00:00|3693539.8  |
|NO4      |hydro          |2021-12-31 18:00:00|3858743.2  |
|NO4      |hydro          |2021-12-31 17:00:00|4052883.5  |
+---------+---------------+-------------------+-----------+
only showing top 5 rows



In [8]:
# === Plots (Saved to figs/ Directory) ===

import os, calendar as _cal
# Set Matplotlib backend to 'Agg' for non-interactive plotting (required in environments like notebooks without a display server)
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from datetime import datetime as _dt, timezone as _tz
from pyspark.sql import functions as F

# Create the output directory for figures if it doesn't exist
os.makedirs("figs", exist_ok=True)

# 1. Pie Chart: Annual Total Production by Group

# Use Spark to calculate the total production for the chosen price area (e.g., NO1)
year_area = (
    cdf.filter(F.col("priceArea") == CHOSEN_AREA)
       .groupBy("productionGroup")
       .agg(F.sum("quantityKwh").alias("totalKwh")) # Sum quantity for each production group
       .orderBy("productionGroup")
)

# Convert the resulting Spark DataFrame to a Pandas DataFrame for Matplotlib plotting
pdf_pie = year_area.toPandas()

# Create and configure the pie chart
plt.figure()
plt.pie(
    pdf_pie["totalKwh"],
    labels=pdf_pie["productionGroup"],
    autopct="%1.1f%%" # Format percentage display
)
plt.title(f"Total Production {YEAR} – {CHOSEN_AREA}") # Set title in English

# Save the pie chart
pie_path = os.path.join("figs", f"pie_total_{YEAR}_{CHOSEN_AREA}.png")
plt.savefig(pie_path, dpi=160, bbox_inches="tight")
plt.close() # Close the figure to free up memory
print("Saved pie chart to:", pie_path)


# 2. Line Chart: Hourly Production for the First Month

# Define UTC boundaries for the first month (e.g., January 2021)
month_start = _dt(YEAR, FIRST_MONTH, 1, 0, 0, 0, tzinfo=_tz.utc)
last_day = _cal.monthrange(YEAR, FIRST_MONTH)[1]
month_end   = _dt(YEAR, FIRST_MONTH, last_day, 23, 0, 0, tzinfo=_tz.utc)

# Filter Spark DataFrame for the chosen area and the first month's time range
jan = (
    cdf.filter((F.col("priceArea") == CHOSEN_AREA) &
               (F.col("startTime") >= F.lit(month_start)) &
               (F.col("startTime") <= F.lit(month_end)))
       .groupBy("startTime", "productionGroup")
       .agg(F.sum("quantityKwh").alias("kwh")) # Aggregate hourly production
)

# Pivot the data: move 'productionGroup' values into new columns, making it wide format
jan_wide = (
    jan.groupBy("startTime")
       .pivot("productionGroup")
       .agg(F.first("kwh"))
       .orderBy("startTime")
)

# Convert to Pandas, setting 'startTime' as the index for time series plotting
pdf_line = jan_wide.toPandas().set_index("startTime")

# Create and configure the line chart
plt.figure(figsize=(11,5))
# Plot each production group as a separate line
for col in pdf_line.columns:
    plt.plot(pdf_line.index, pdf_line[col], label=col)
    
# Set English titles and labels
month_name = month_start.strftime('%B %Y')
plt.title(f"Hourly Production – {CHOSEN_AREA}, {month_name}")
plt.xlabel("Time (UTC)")
plt.ylabel("kWh")
plt.legend(title="Production Group")
plt.tight_layout()

# Save the line chart
line_path = os.path.join("figs", f"line_{YEAR}_{CHOSEN_AREA}_{FIRST_MONTH:02d}.png")
plt.savefig(line_path, dpi=160, bbox_inches="tight")
plt.close()
print("Saved line chart to:", line_path)

Saved pie chart to: figs/pie_total_2021_NO1.png
Saved line chart to: figs/line_2021_NO1_01.png


In [9]:
# Write to MongoDB

from pyspark.sql import functions as F

# 1. Prepare Data for MongoDB
# MongoDB uses the '_id' field as the primary key. We create a compound, unique
# identifier by concatenating the natural primary key elements: Price Area,
# Production Group, and the exact timestamp.
cdf_out = (
    cdf.withColumn(
        "_id",
        F.concat_ws("#",
            F.col("priceArea"),
            F.col("productionGroup"),
            # Format the startTime timestamp into a consistent ISO 8601 string with UTC offset
            F.date_format("startTime", "yyyy-MM-dd'T'HH:mm:ssXXX")
        )
    )
)

# Write to MongoDB Atlas
(cdf_out.write.format("mongodb")
    .mode("overwrite")   # Use 'overwrite' to replace all existing documents in the collection
    # .mode("append")    # Use 'append' if you don't want to overwrite
    .option("uri", MONGO_URI)
    .option("database", MONGO_DB)
    .option("collection", MONGO_COLL)
    .save())

print(f"Written to MongoDB → {MONGO_DB}.{MONGO_COLL}")

Written to MongoDB → elhub.production_2021_hourly_by_group
